# M4.8 · Modern CNN blokları — Inception, ayrılabilir konvolüsyon, SE, grup konvolüsyonu, LRN

Tam bağlı ağlar ile CNN'lerin verimlilik karşılaştırması (ders alıştırması 5: eşit parametreli FC ve CNN) ve LeNet'ten EfficientNet'e kadar CNN bloklarının arkasındaki hesap: AlexNet'in yerel tepki normalizasyonu (LRN), GoogLeNet'in Inception modülü ve 1×1 "darboğaz" konvolüsyonları, Xception/MobileNet'in derinlemesine ayrılabilir konvolüsyonu, ResNeXt'in grup konvolüsyonu, SENet'in sıkıştır-uyar (squeeze-and-excitation) kanal dikkati, küresel ortalama havuzlama ve sınıf aktivasyon haritaları, MobileNetV2'nin ters artık bloğu ve EfficientNet'in bileşik ölçeklemesi. Saf Python'da parametre/FLOP sayaçları, NumPy'da blokların vektörel uygulamaları ve eşdeğerlikleri, PyTorch'ta modüller ve kaydırılmış rakamlarda FC vs CNN, sonda FLOP ile gerçek sürenin neden ayrıştığı.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-8-cnn-blocks.html](https://m-gonul.github.io/pytorch-lab/m4-8-cnn-blocks.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_8_cnn_blocks.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef conv(k, ci, co, g=1):\n    return {\"type\": \"conv\", \"k\": k, \"cin\": ci, \"cout\": co, \"groups\": g, \"bias\": True}\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    P = {\"type\": \"pool\", \"k\": 2}\n\n    def ex5():\n        cnn2 = [conv(7, 1, 16), P, conv(3, 16, 32), P, conv(3, 32, 64), P, {\"type\": \"flatten\"}, {\"type\": \"linear\", \"in\": 576, \"out\": 64}, {\"type\": \"linear\", \"in\": 64, \"out\": 10}]\n        fc3 = [{\"type\": \"flatten\"}, {\"type\": \"linear\", \"in\": 784, \"out\": 74}, {\"type\": \"linear\", \"in\": 74, \"out\": 50}, {\"type\": \"linear\", \"in\": 50, \"out\": 10}]\n        fc1 = [{\"type\": \"flatten\"}, {\"type\": \"linear\", \"in\": 784, \"out\": 300}, {\"type\": \"linear\", \"in\": 300, \"out\": 100}, {\"type\": \"linear\", \"in\": 100, \"out\": 10}]\n        bn = lambda c: {\"type\": \"bn\", \"c\": c}\n        cnn4 = [conv(7, 1, 64), bn(64), P, bn(64), conv(5, 64, 128), bn(128), conv(3, 128, 128), bn(128), P, bn(128), conv(3, 128, 256), bn(256), conv(3, 256, 256), bn(256), P,\n                {\"type\": \"flatten\"}, {\"type\": \"linear\", \"in\": 2304, \"out\": 128}, {\"type\": \"linear\", \"in\": 128, \"out\": 64}, {\"type\": \"linear\", \"in\": 64, \"out\": 10}]\n        rows = []\n        for name, m in ((\"FC 784-300-100-10\", fc1), (\"CNN (alıştırma 2)\", cnn2), (\"FC 784-74-50-10\", fc3), (\"BN'li büyük CNN (alıştırma 4)\", cnn4)):\n            p, mac, shp = need(ns, \"count\")(m, (1, 28, 28))\n            rows.append([name, f\"{p:,}\".replace(\",\", \" \"), f\"{mac / 1e6:.2f} M\"])\n        ensure(ns[\"count\"](cnn2, (1, 28, 28))[0] == 61514 and ns[\"count\"](fc3, (1, 28, 28))[0] == 62350 and ns[\"count\"](cnn4, (1, 28, 28))[0] == 1546954, \"parametre sayıları\")\n        ensure(ns[\"count\"](cnn2[:4], (1, 28, 28))[2] == (32, 7, 7), \"çıkış şekli\")\n        R_.table(\"Ders alıştırması 5'in modelleri\", [\"model\", \"parametre\", \"MAC/örnek\"], rows,\n                 note=\"Eşit parametreli FC ve CNN'de hesap çok farklı: CNN her ağırlığı yüzlerce konumda yeniden kullanır (5.7 M MAC vs 0.06 M). Parametre ≠ hesap.\")\n    R_.test(\"sayaçlar\", ex5)\n\n    def blocks():\n        p, c = need(ns, \"inception_params\")(192, 64, 96, 128, 16, 32, 32)\n        ensure(p == 192 * 64 + 192 * 96 + 9 * 96 * 128 + 192 * 16 + 25 * 16 * 32 + 192 * 32 and c == 256, \"Inception 3a\")\n        dw = need(ns, \"count\")([conv(3, 128, 128, 128), conv(1, 128, 256)], (128, 14, 14))\n        st = ns[\"count\"]([conv(3, 128, 256)], (128, 14, 14))\n        ensure(dw[0] < st[0] / 8, \"ayrılabilir ~9× az parametre\")\n        close(need(ns, \"lrn\")([0, 0, 2.0, 2.0, 0], 2, 5, 1e-4, 0.75)[2], 2 / (2 + 1e-4 * 8) ** 0.75, what=\"LRN\")\n    R_.test(\"bloklar ve LRN\", blocks)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: [max_loops(src, 0, func=f) for f in (\"conv2d\", \"depthwise\", \"pointwise\", \"grouped_conv\", \"se_block\", \"lrn\")])\n    rng = np.random.default_rng(0)\n\n    def eq():\n        x = rng.normal(size=(8, 9, 10))\n        dw = rng.normal(size=(8, 3, 3))\n        pw = rng.normal(size=(6, 8))\n        full = np.zeros((8, 8, 3, 3))\n        full[np.arange(8), np.arange(8)] = dw\n        close(need(ns, \"depthwise\")(x, dw), need(ns, \"conv2d\")(x, full), atol=1e-10, what=\"derinlemesine = köşegen tam konv.\")\n        sep = ns[\"pointwise\"](ns[\"depthwise\"](x, dw), pw)\n        close(sep, ns[\"conv2d\"](x, np.einsum(\"oc,cuv->ocuv\", pw, dw)), atol=1e-10, what=\"ayrılabilir = rank-yapılı tam konv.\")\n        close(need(ns, \"pointwise\")(x, pw), ns[\"conv2d\"](x, pw[:, :, None, None]), atol=1e-10, what=\"1×1 konv. = noktasal\")\n        wg = rng.normal(size=(8, 2, 3, 3))\n        g = need(ns, \"grouped_conv\")(x, wg, 4)\n        blk = np.zeros((8, 8, 3, 3))\n        for gi in range(4):\n            blk[2 * gi:2 * gi + 2, 2 * gi:2 * gi + 2] = wg[2 * gi:2 * gi + 2]\n        close(g, ns[\"conv2d\"](x, blk), atol=1e-10, what=\"grup konv. = blok köşegen tam konv.\")\n        close(ns[\"grouped_conv\"](x, dw[:, None], 8), ns[\"depthwise\"](x, dw), atol=1e-10, what=\"g = C → derinlemesine\")\n    R_.test(\"eşdeğerlikler\", eq)\n\n    def se_lrn():\n        x = rng.normal(size=(16, 5, 5))\n        W1, b1, W2, b2 = rng.normal(size=(4, 16)), rng.normal(size=4), rng.normal(size=(16, 4)), rng.normal(size=16)\n        y = need(ns, \"se_block\")(x, W1, b1, W2, b2)\n        r = y / x\n        ensure(np.allclose(r, r[:, :1, :1]) and (r > 0).all() and (r < 1).all(), \"SE kanal başına (0, 1) ölçek\")\n        a = rng.normal(size=(7, 3, 3)) * 5\n        ref = np.stack([a[i] / (2 + 1e-4 * (a[max(0, i - 2):i + 3] ** 2).sum(0)) ** 0.75 for i in range(7)])\n        close(need(ns, \"lrn\")(a), ref, rtol=1e-10, what=\"LRN\")\n    R_.test(\"SE ve LRN\", se_lrn)\n",
 "d": "import torch\nimport torch.nn as nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def mods():\n        x = torch.randn(2, 32, 9, 9)\n        ds = need(ns, \"DepthwiseSeparable\")(32, 64)\n        ensure(tuple(ds(x).shape) == (2, 64, 9, 9), \"şekil\")\n        ensure(sum(p.numel() for p in ds.parameters()) == 9 * 32 + 32 * 64 + 64, \"parametre: 9C + C·C' + C'\")\n        se = need(ns, \"SE\")(32, 8)\n        y = se(x)\n        r = y / x\n        ensure(tuple(y.shape) == x.shape and torch.allclose(r, r[:, :, :1, :1].expand_as(r), atol=1e-5), \"SE kanal ölçeklemesi\")\n        inc = need(ns, \"Inception\")(192, 64, 96, 128, 16, 32, 32)\n        ensure(tuple(inc(torch.randn(1, 192, 28, 28)).shape) == (1, 256, 28, 28), \"Inception 3a: 256 kanal\")\n        lrn = nn.LocalResponseNorm(5, alpha=5e-4, beta=0.75, k=2.0)\n        a = torch.randn(1, 7, 3, 3) * 5\n        ref = torch.stack([a[0, i] / (2 + 1e-4 * (a[0, max(0, i - 2):i + 3] ** 2).sum(0)) ** 0.75 for i in range(7)])\n        close(lrn(a)[0].numpy(), ref.numpy(), rtol=1e-5, what=\"nn.LocalResponseNorm(α/n kuralı) = AlexNet LRN (α = 5·10⁻⁴/5)\")\n    R_.test(\"bloklar\", mods)\n\n    def ex5():\n        M = need(ns, \"ex5_models\")()\n        cnt = {k: sum(p.numel() for p in m.parameters() if p.requires_grad) for k, m in M.items()}\n        ensure(cnt[\"cnn2\"] == 61514 and cnt[\"fc3\"] == 62350 and cnt[\"fc1\"] == 266610 and cnt[\"cnn4\"] == 1546954, \"parametre sayıları\")\n        X, y = need(ns, \"shifted_digits\")(3, 0)\n        Xt, yt = ns[\"shifted_digits\"](1, 1)\n        ensure(tuple(X.shape) == (5391, 1, 28, 28) and float(X.sum(dim=(1, 2, 3)).min()) > 0, \"kaydırılmış rakamlar\")\n        acc = {k: need(ns, \"train_eval\")(M[k], X, y, Xt, yt, 8) for k in (\"fc1\", \"cnn2\", \"fc3\")}\n        ensure(acc[\"cnn2\"] > acc[\"fc3\"] + 0.05 and acc[\"cnn2\"] > acc[\"fc1\"], \"eşit bütçede CNN, 4× büyük FC'yi de geçmeli\")\n        R_.table(\"Kaydırılmış rakamlar (28×28), 8 epoch\", [\"model\", \"parametre\", \"test doğruluğu\"], [[k, cnt[k], f\"{acc[k]:.3f}\"] for k in acc],\n                 note=\"Rakam tuvalin her yerinde olabilir: FC her konum için desenleri ayrı öğrenmeli; CNN aynı filtreyi her yerde uygular (eşdeğerlik) ve havuzlama küçük kaymaları emer. Model 4 (BN'li büyük CNN) ödev ölçeğinde: daha uzun eğitimle en iyisi ama 25× parametre.\")\n    R_.test(\"ders alıştırması 5\", ex5)\n",
 "e": "from labcheck import need, ensure, fmt_time\n\n\ndef check(ns, R_):\n    def t():\n        r = need(ns, \"time_blocks\")(128, 56)\n        ensure(set(r) == {\"3×3 standart\", \"3×3 ayrılabilir\", \"3×3 grup (g = 8)\", \"1×1\"}, \"dört blok\")\n        st, sp = r[\"3×3 standart\"], r[\"3×3 ayrılabilir\"]\n        ensure(sp[0] < st[0] / 7, \"ayrılabilir MAC ~8× az\")\n        ensure(sp[2] < st[2], \"ayrılabilirin GMAC/s'si (verimi) düşük olmalı\")\n        R_.table(\"C = 128, 56×56, batch 8 (CPU)\", [\"blok\", \"MAC\", \"süre\", \"GMAC/s\"], [[k, f\"{m / 1e9:.2f} G\", fmt_time(t), f\"{v:.1f}\"] for k, (m, t, v) in r.items()],\n                 note=f\"Ayrılabilir blok {st[0] / sp[0]:.1f}× az MAC yapar ama süre yalnız {st[1] / sp[1]:.1f}× kısalır: derinlemesine kısım kanal başına küçük, bellek-bağımlı bir işlemdir. Mobil donanım ve özel çekirdekler bu farkı kapatmaya çalışır.\")\n    R_.test(\"FLOP ve süre\", t)\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngüsüz\", lambda: (max_loops(src, 0, func=\"gap_head\"), max_loops(src, 0, func=\"cam\")))\n\n    def t():\n        rng = np.random.default_rng(0)\n        F = np.maximum(0, rng.normal(size=(64, 7, 7)))\n        W = rng.normal(size=(10, 64))\n        b = rng.normal(size=10)\n        logits = need(ns, \"gap_head\")(F, W, b)\n        for c in (0, 3, 9):\n            close(need(ns, \"cam\")(F, W, c).mean(), logits[c] - b[c], atol=1e-10, what=f\"CAM ortalaması = logit (sınıf {c})\")\n        F2 = np.zeros((2, 7, 7))\n        F2[0, 1, 5] = 10\n        F2[1, 5, 1] = 10\n        m = ns[\"cam\"](F2, np.array([[1.0, 0.0], [0.0, 1.0]]), 0)\n        ensure(np.unravel_index(m.argmax(), m.shape) == (1, 5), \"CAM sınıfın kanıtını konumlandırır\")\n        a, g = need(ns, \"params_flatten_vs_gap\")(512, 7, 7, 1000, 4096)\n        R_.table(\"VGG-16 başı vs GAP başı (512×7×7 → 1000 sınıf)\", [\"baş\", \"parametre\"], [[\"düzleştir + FC 4096 + FC 1000\", f\"{a:,}\".replace(\",\", \" \")], [\"GAP + FC 1000\", f\"{g:,}\".replace(\",\", \" \")]],\n                 note=\"GoogLeNet ve ResNet'ten beri sınıflandırma başı GAP'tir: yüz milyonlarca parametreden kurtarır, girdi boyutundan bağımsızdır ve CAM ile 'model nereye baktı'yı ücretsiz verir.\")\n    R_.test(\"GAP ve CAM\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        ensure(need(ns, \"inverted_residual_cost\")(32, 28, 6) == (32 * 192 * 2 + 9 * 192, 28 * 28 * (32 * 192 * 2 + 9 * 192)), \"ters artık\")\n        ensure(need(ns, \"bottleneck_cost\")(256, 14, 4) == (256 * 64 * 2 + 9 * 64 * 64, 14 * 14 * (256 * 64 * 2 + 9 * 64 * 64)), \"darboğaz\")\n        rows = []\n        for t_ in (1, 2, 4, 6, 8):\n            p, m = ns[\"inverted_residual_cost\"](64, 28, t_)\n            rows.append([t_, p, f\"{m / 1e6:.1f} M\"])\n        rows.append([\"ResNet darboğazı (r = 4)\", ns[\"bottleneck_cost\"](64, 28, 4)[0], f\"{ns['bottleneck_cost'](64, 28, 4)[1] / 1e6:.1f} M\"])\n        R_.table(\"c = 64, 28×28: ters artık bloğun genişleme oranı t\", [\"t\", \"parametre\", \"MAC\"], rows,\n                 note=\"ResNet darboğazı kanalları ortada daraltır (geniş → dar → geniş); MobileNetV2 tersini yapar (dar → geniş → dar) ve pahalı uzamsal işi ucuz derinlemesine konvolüsyona bırakır. Maliyet t ile doğrusal; son 1×1'den sonra ReLU yok (doğrusal darboğaz: dar uzayda bilgi kaybını önler).\")\n    R_.test(\"ters artık\", t)\n",
 "x3": "from labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"flops_factor\")(1.2, 1.1, 1.15, 1), 1.2 * 1.21 * 1.3225, what=\"B0 katsayıları ≈ 1.92\")\n        cands = need(ns, \"grid_candidates\")(0.05)\n        ensure((1.2, 1.1, 1.15) in cands and all(abs(a * b * b * g * g - 2) <= 0.1 for a, b, g in cands), \"kısıtı sağlayanlar\")\n        rows = []\n        for phi in range(0, 8):\n            d, w, r = need(ns, \"scale_model\")((18, 32, 224), 1.2, 1.1, 1.15, phi)\n            rows.append([f\"B{phi}\", d, w, r, f\"{ns['flops_factor'](1.2, 1.1, 1.15, phi):.1f}×\"])\n        R_.table(\"EfficientNet-B0'dan bileşik ölçekleme (α, β, γ) = (1.2, 1.1, 1.15)\", [\"model\", \"katman\", \"kök genişliği\", \"çözünürlük\", \"FLOP\"], rows,\n                 note=\"Yalnız derinliği ya da yalnız genişliği büyütmek çabuk doyar (makale Şekil 3); üçünü dengeli büyütmek daha büyük girdinin daha büyük receptive field'a ve daha çok kanala ihtiyaç duyduğu sezgisini kodlar. (Gerçek B1–B7 çözünürlükleri biraz farklı yuvarlanmıştır.)\")\n    R_.test(\"bileşik ölçekleme\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m4_8_cnn_blocks as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**FC mi CNN mi** (ders alıştırması 5, Fashion-MNIST): (1) önceki alıştırmanın FC ağı; (2) CNN: 7×7×16 + havuz, 3×3×32 + havuz, 3×3×64 + havuz, düzleştir, FC 64, softmax 10; (3) FC 784-74-50-10 — CNN ile **aynı parametre bütçesi**; (4) BatchNorm'lu daha büyük bir CNN. Gözlem: eşit parametreyle CNN çok daha iyi genelleşir — yerellik + ağırlık paylaşımı güçlü bir önseldir.

**Klasik mimariler:** LeNet-5 (1998), **AlexNet** (2012: ReLU, dropout, veri artırma, **LRN** — güçlü aktivasyonların komşu kanalları bastırması: $b_i=a_i/(k+\alpha\sum_{j\in N(i)}a_j^2)^\beta$), **GoogLeNet** (Inception modülleri: paralel 1×1, 3×3, 5×5, havuz dalları; pahalı dallardan önce **1×1 darboğaz**), VGG (3×3 yığınları), ResNet (artık bağlantı, M4.2), **Xception** (derinlemesine ayrılabilir: önce her kanalı ayrı uzamsal filtrele, sonra 1×1 ile karıştır), **ResNeXt** (grup konvolüsyonu: "kardinalite"), **SENet** (kanalları küresel ortalamayla "sıkıştır", iki FC + sigmoid ile kanal ağırlıklarını öğren, öznitelik haritasını yeniden ölçekle), DenseNet (birleştirme), MobileNet/MobileNetV2 (ters artık: 1×1 genişlet → derinlemesine → 1×1 doğrusal daralt), **EfficientNet** (derinlik, genişlik ve çözünürlüğü birlikte ölçekle).

**Sayım:** standart konv. $K^2C_{in}C_{out}$ parametre, her çıktı pikseli için o kadar MAC. Derinlemesine $K^2C$ + noktasal $C_{in}C_{out}$ → oran $\approx1/C_{out}+1/K^2$. Grup konvolüsyonu (g grup) parametreyi g'ye böler. **Küresel ortalama havuzlama** (GAP), düzleştirme + büyük FC katmanını kaldırır; son katman ağırlıkları uzamsal haritalarla birleşince sınıf aktivasyon haritası (CAM) verir.

## 1 · Elle

CNN: 800 + 4640 + 18 496 + (3·3·64 = 576 → 64: 36 928) + 650 = 61 514 (28 → 14 → 7 → 3). FC: 58 090 + 3 750 + 510 = 62 350 — alıştırmanın kurgusu: eşit bütçe.

Darboğaz: 153 600 → 3 072 + 12 800 = 15 872 (~9.7×). Ayrılabilir: 294 912 / (1 152 + 32 768) = 8.69. SE: 256·16 + 16 + 16·256 + 256 = 8 464 (konvolüsyonun yanında küçük).

LRN: $2/2.0008^{0.75}=1.1889$ — komşu kanallar küçükken etkisi bir sabit ölçek; büyük aktivasyonlar komşularını bastırır. BatchNorm geldikten sonra terk edildi.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Ders alıştırması 5 (Fashion-MNIST 28×28): eğitilebilir parametre**

- `cnn2`: CNN: 7×7×16 → havuz → 3×3×32 → havuz → 3×3×64 → havuz → FC 64 → 10 (same dolgu)
- `fc3`: FC: 784 → 74 → 50 → 10

**Bloklar (bias yok)**

- `inc`: 5×5 konv. 192 → 32 kanal: parametre
- `incb`: Önce 1×1 ile 192 → 16, sonra 5×5 16 → 32: parametre
- `sep`: 3×3 konv. 128 → 256: standart / derinlemesine ayrılabilir oranı
- `se`: SE, C = 256, r = 16: parametre (bias'lar dahil)

**LRN (AlexNet): $b=a/(k+\alpha\sum a^2)^\beta$**

- `lrn`: a = 2, komşu kanallarda Σa² = 8, k = 2, α = 10⁻⁴, β = 0.75

In [ ]:
cevaplar = dict(
    cnn2=None,
    fc3=None,
    inc=None,
    incb=None,
    sep=None,
    se=None,
    lrn=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"cnn2":[61514,0],"fc3":[62350,0],"inc":[153600,0],"incb":[15872,0],"sep":[8.69433962264151,0.000001],"se":[8464,0],"lrn":[1.1888504776892002,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python sayaçları. Katman tanımları sözlük: `{"type": "conv", "k": 3, "cin": 16, "cout": 32, "groups": 1, "bias": True}`, `{"type": "pool", "k": 2}`, `{"type": "linear", "in": 576, "out": 64}`, `{"type": "bn", "c": 64}`, `{"type": "flatten"}`.

- `count(layers, in_shape)` → `(parametre, MAC, çıkış şekli)`; `in_shape = (C, H, W)`; konv. "same" dolgu, adım 1; havuz adım = k (taban); BN 2C eğitilebilir parametre; MAC: konv. $H W\,K^2C_{in}C_{out}/g$, doğrusal in·out.
- `inception_params(cin, c1, c3r, c3, c5r, c5, pp)` — GoogLeNet modülü (bias yok): 1×1 dalı c1; 1×1 c3r → 3×3 c3; 1×1 c5r → 5×5 c5; 3×3 havuz → 1×1 pp. Çıkış kanal sayısı ile birlikte.
- `lrn(a, k=2, n=5, alpha=1e-4, beta=0.75)` — `a` kanal listesi (tek piksel), AlexNet LRN (pencere n, α bölünmeden).

In [ ]:
%%writefile stage_b.py
def count(layers, in_shape):
    ...


def inception_params(cin, c1, c3r, c3, c5r, c5, pp):
    ...


def lrn(a, k=2, n=5, alpha=1e-4, beta=0.75):
    ...


cnn2 = [{"type": "conv", "k": 7, "cin": 1, "cout": 16, "groups": 1, "bias": True}, {"type": "pool", "k": 2}]
print(count(cnn2, (1, 28, 28)))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz. Görüntüler `(C, H, W)`, "same" dolgu, adım 1.

- `conv2d(x, w)` — `w (Cout, Cin, K, K)` (`sliding_window_view` + `einsum`).
- `depthwise(x, w)` — `w (C, K, K)`; `pointwise(x, w)` — `w (Cout, Cin)`.
- `grouped_conv(x, w, g)` — `w (Cout, Cin/g, K, K)`.
- `se_block(x, W1, b1, W2, b2)` — GAP → ReLU(W1 · + b1) → sigmoid(W2 · + b2) → kanal ölçekleme.
- `lrn(x, k=2, n=5, alpha=1e-4, beta=0.75)` — kanal ekseninde kayan kareler toplamı (kümülatif toplamla).

In [ ]:
%%writefile stage_c.py
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def conv2d(x, w):
    ...


def depthwise(x, w):
    ...


def pointwise(x, w):
    ...


def grouped_conv(x, w, g):
    ...


def se_block(x, W1, b1, W2, b2):
    ...


def lrn(x, k=2, n=5, alpha=1e-4, beta=0.75):
    ...

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch.

- `DepthwiseSeparable(cin, cout, k=3)`, `SE(c, r=16)`, `Inception(cin, c1, c3r, c3, c5r, c5, pp)` (her dal Conv + ReLU; çıktılar kanal ekseninde birleşir).
- `ex5_models()` → `{"fc1", "cnn2", "fc3", "cnn4"}`: alıştırmanın dört modeli (28×28 girdi; cnn4: 7×7×64, BN, havuz, BN, 5×5×128, BN, 3×3×128, BN, havuz, BN, 3×3×256, BN, 3×3×256, BN, havuz, FC 128, 64, 10).
- `shifted_digits(n_rep, seed)` — sklearn rakamlarını 20×20'ye büyütüp 28×28 tuvalde rastgele konuma koy (Fashion-MNIST yerine çevrimdışı vekil; konum çeşitliliği CNN'in önselini sınar).
- `train_eval(model, X, y, Xt, yt, epochs=8)` → test doğruluğu (Adam 1e-3, batch 64).

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits


class DepthwiseSeparable(nn.Module):
    def __init__(self, cin, cout, k=3):
        super().__init__()
        ...

    def forward(self, x):
        ...


class SE(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        ...

    def forward(self, x):
        ...


class Inception(nn.Module):
    def __init__(self, cin, c1, c3r, c3, c5r, c5, pp):
        super().__init__()
        ...

    def forward(self, x):
        ...


def ex5_models():
    ...


def shifted_digits(n_rep, seed):
    ...


def train_eval(model, X, y, Xt, yt, epochs=8):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

- `time_blocks(C, H, reps=5)` → sözlük: standart 3×3 konv., derinlemesine ayrılabilir, grup (g = 8) ve 1×1 konvolüsyonun MAC sayısı, süresi ve elde edilen GMAC/s'si (`C → C` kanal, `H × H`, batch 8, CPU).

In [ ]:
%%writefile stage_e.py
import time
import torch
import torch.nn as nn


def time_blocks(C, H, reps=5):
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Küresel ortalama havuzlama ve sınıf aktivasyon haritası (CAM)

- `gap_head(F, W, b)` — `F (K, H, W)` son konv. haritaları, `W (classes, K)`: GAP → logit'ler.
- `cam(F, W, c)` — sınıf c için $\sum_kW_{ck}F_k$ haritası; doğrulama: ortalaması logit − bias'a eşit.
- `params_flatten_vs_gap(K, H, W, classes, hidden)` → (düzleştir + FC gizli + çıkış, GAP + çıkış).

In [ ]:
%%writefile extra_x1.py
import numpy as np


def gap_head(F, W, b):
    ...


def cam(F, W, c):
    ...


def params_flatten_vs_gap(K, H, W, classes, hidden):
    ...

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · MobileNetV2 ters artık bloğu: genişleme oranının bedeli

- `inverted_residual_cost(c, H, t, k=3)` → `(parametre, MAC)`: 1×1 genişletme c → tc, k×k derinlemesine, 1×1 doğrusal daralt tc → c (bias/BN yok).
- `bottleneck_cost(c, H, r, k=3)` → ResNet darboğazı: 1×1 c → c/r, 3×3 c/r → c/r, 1×1 c/r → c.

In [ ]:
%%writefile extra_x2.py
def inverted_residual_cost(c, H, t, k=3):
    ...


def bottleneck_cost(c, H, r, k=3):
    ...


print(inverted_residual_cost(32, 28, 6))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · EfficientNet bileşik ölçekleme

EfficientNet: derinlik $d=\alpha^\phi$, genişlik $w=\beta^\phi$, çözünürlük $r=\gamma^\phi$, FLOP ∝ $d\,w^2r^2$; kısıt $\alpha\beta^2\gamma^2\approx2$ (her φ adımı FLOP'u ikiye katlar).

- `flops_factor(alpha, beta, gamma, phi)`.
- `grid_candidates(step=0.05)` → kısıtı ±%5 içinde sağlayan $(\alpha,\beta,\gamma)\in[1,2]^3$ üçlüleri.
- `scale_model(base, alpha, beta, gamma, phi)` — `base = (derinlik, genişlik, çözünürlük)` → yuvarlanmış yeni üçlü.

In [ ]:
%%writefile extra_x3.py
def flops_factor(alpha, beta, gamma, phi):
    ...


def grid_candidates(step=0.05):
    ...


def scale_model(base, alpha, beta, gamma, phi):
    ...


print(flops_factor(1.2, 1.1, 1.15, 1))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Chollet (2017): Xception

[Chollet — Xception: Deep Learning with Depthwise Separable Convolutions (CVPR 2017)](https://arxiv.org/abs/1610.02357)

Makalenin "Inception modülü = düzenli konvolüsyon ile derinlemesine ayrılabilir konvolüsyon arasındaki bir spektrum" savını deneyle sına: CIFAR-10'da aynı derinlik/genişlikte (a) standart konvolüsyonlu, (b) segment sayısı 1, 2, 4, 8 olan Inception-benzeri (grup konvolüsyonu ile), (c) tam derinlemesine ayrılabilir bloklar kur; parametre/FLOP/doğruluk tablosu çıkar. Makaledeki "ara doğrusalsızlık yok" bulgusunu (Şekil 10) de test et.


**Adımlar**

1. d aşamasındaki DepthwiseSeparable ve grup konvolüsyonu modüllerini kullan.

**Kabul ölçütleri**

- Ayrılabilir sürüm, benzer doğrulukta belirgin daha az parametre.
- Derinlemesine ve noktasal arasında ReLU olmaması daha iyi ya da eşit.

## Makale ödevi: Hu ve ark. (2018): Sıkıştır-uyar ağları

[Hu, Shen, Sun — Squeeze-and-Excitation Networks (CVPR 2018)](https://arxiv.org/abs/1709.01507)

Küçük bir ResNet'e (M4.2) SE blokları ekle, r ∈ {4, 8, 16, 32} için parametre artışı ve CIFAR-10 doğruluğunu ölç; makalenin Şekil 6'sındaki gibi farklı sınıflarda farklı katmanlardaki uyarım (excitation) dağılımlarını çiz ve derin katmanların sınıfa özgü hâle geldiğini göster.


**Adımlar**

1. Uyarım vektörlerini hook ile topla; sınıf başına ortalamasını al.

**Kabul ölçütleri**

- SE, %1'den az ek parametreyle doğruluğu artırır ya da korur.
- Son katmanlarda sınıflar arası uyarım farkı ilk katmanlardan büyük.

## Kaynaklar

- [Géron — Hands-On Machine Learning (3. baskı), Bölüm 14](https://github.com/ageron/handson-ml3) — Apache-2.0 (kod)
- [Szegedy ve ark. — Going Deeper with Convolutions (CVPR 2015)](https://arxiv.org/abs/1409.4842) — arXiv
- [Chollet — Xception: Deep Learning with Depthwise Separable Convolutions (CVPR 2017)](https://arxiv.org/abs/1610.02357) — arXiv
- [Hu, Shen, Sun — Squeeze-and-Excitation Networks (CVPR 2018)](https://arxiv.org/abs/1709.01507) — arXiv
- [Krizhevsky, Sutskever, Hinton — ImageNet Classification with Deep CNNs (NeurIPS 2012)](https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html) — ©
- [Xie ve ark. — Aggregated Residual Transformations (ResNeXt, CVPR 2017)](https://arxiv.org/abs/1611.05431) — arXiv
- [Sandler ve ark. — MobileNetV2: Inverted Residuals and Linear Bottlenecks (CVPR 2018)](https://arxiv.org/abs/1801.04381) — arXiv
- [Tan & Le — EfficientNet: Rethinking Model Scaling for CNNs (ICML 2019)](https://arxiv.org/abs/1905.11946) — arXiv
- [Zhou ve ark. — Learning Deep Features for Discriminative Localization (CAM, CVPR 2016)](https://arxiv.org/abs/1512.04150) — arXiv